# 🚀 DataMate — Zalo AI Challenge 2026: Public Test Inference

Notebook này giúp bạn chạy pipeline suy luận trên tập **`public_test`** (1,000 câu hỏi) bằng **GPU Colab** (T4 / L4 / A100) và xuất ra file **`submission.jsonl`** chuẩn để nộp lên cổng Zalo AI Challenge.

> **Lưu ý:** Hãy bật GPU trước khi chạy: vào **Runtime** → **Change runtime type** → chọn **T4 GPU** hoặc **A100 GPU**.

In [6]:
# 1. Kiểm tra thông tin GPU
!nvidia-smi

Wed Oct 28 12:23:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q duckdb>=1.1.0 pandas>=2.2.0 openpyxl python-calamine
!pip install -q transformers>=4.48.0 accelerate>=1.0.0 bitsandbytes>=0.43.0
!pip install -q tqdm loguru unidecode regex

### 3. Chuẩn bị Mã nguồn và Dữ liệu

Bạn có thể chọn 1 trong 2 cách:
- **Cách A:** Clone repo Git hoặc kết nối Google Drive đã có thư mục `public_test` và `src/`.
- **Cách B:** Nếu chạy trực tiếp, upload folder `public_test.zip` và `src.zip` lên cột File bên trái rồi giải nén.

In [9]:
# Tùy chọn: Mount Google Drive nếu lưu data trên Drive
from google.colab import drive
drive.mount('/content/drive')

# Trỏ về thư mục chứa source code và public_test (thay đổi đường dẫn nếu cần)
%cd /content/drive/MyDrive/ZALO_AI_Challenge

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ZALO_AI_Challenge


In [16]:
# Giải nén public_test.zip (nếu bạn tải file zip lên)
!unzip -q -o public_test.zip
# !unzip -q -o src.zip  # Mở dòng này nếu bạn cũng nén src thành zip

# Liệt kê các file/folder đang có trên máy ảo Colab
!ls -la

unzip:  cannot find or open public_test.zip, public_test.zip.zip or public_test.zip.ZIP.
total 32
drwx------  4 root root  4096 Oct 28 12:16 .
drwx------ 33 root root  4096 Oct 28 12:14 ..
drwx------  2 root root  4096 Oct 28 12:17 public_test
-rw-------  1 root root 16320 Oct 28 12:11 run_public_test.py
drwx------  2 root root  4096 Oct 28 12:17 src


In [27]:
# Kiểm tra các thư mục bắt buộc phải có
import os
print("public_test có sẵn:", os.path.exists("public_test"))
print("src/ có sẵn:", os.path.exists("src"))
print("run_public_test.py có sẵn:", os.path.exists("run_public_test.py"))

public_test có sẵn: True
src/ có sẵn: True
run_public_test.py có sẵn: True


### 4. Chạy thử nghiệm trên 5 câu đầu tiên (Sanity Check)
Chạy thử 5 câu để kiểm tra model, DuckDB và code execution có hoạt động trơn tru không.

In [29]:
!python run_public_test.py \
    --data_dir public_test \
    --output submission_sample.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --load_in_4bit \
    --limit 5

=== Starting DataMate Inference ===
[*] Using data directory: /content/drive/MyDrive/ZALO_AI_Challenge/public_test
[*] Initializing model: Qwen/Qwen2.5-Coder-7B-Instruct on cuda (4-bit: True)
Loading weights: 100% 339/339 [00:04<00:00, 79.50it/s]
[+] Model loaded successfully!
[*] Processing first 5 questions only.
Inferring: 100% 5/5 [01:46<00:00, 21.24s/it]

[+] Inference completed successfully!
[+] Output saved to: submission_sample.jsonl
[+] Logs saved to: result/logs


In [30]:
# Xem kết quả 5 câu chạy thử
!cat submission_sample.jsonl

{"id": "q_008fdebb6ec9", "type": "abstain"}
{"id": "q_009187cd2960", "type": "abstain"}
{"id": "q_00bf8df3ea0b", "type": "abstain"}
{"id": "q_00c4e24fa228", "type": "abstain"}
{"id": "q_013b4ff35811", "type": "abstain"}


### 5. Chạy toàn bộ 1,000 câu hỏi trên Public Test
Lệnh này sẽ quét toàn bộ 1,000 câu và ghi liên tục vào file `submission.jsonl`.

In [ ]:
!python run_public_test.py \
    --data_dir public_test \
    --output submission.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --load_in_4bit

### 6. Kiểm tra & Thống kê File `submission.jsonl`
Kiểm tra đủ số lượng 1,000 dòng và phân bố các type trước khi nộp.

In [ ]:
import json
from collections import Counter

lines = []
with open("submission.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            lines.append(json.loads(line))

print(f"Tổng số dòng đã sinh: {len(lines)} / 1000")
types = Counter([x.get("type") for x in lines])
print("Phân bố các loại đáp án:")
for t, c in types.most_common():
    print(f"  {t}: {c} ({c/len(lines)*100:.1f}%)")

In [ ]:
# 7. Tải file submission.jsonl về máy để nộp lên cổng Zalo AI Challenge
from google.colab import files
files.download('submission.jsonl')